

- [LCEL+structured outputの使い方](https://qiita.com/ssc-ymuramatsu/items/120e87d4a4751ab608f9)

In [ ]:
import os
import time
from logging import DEBUG, INFO, StreamHandler, getLogger
from typing import List

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAI
from pydantic import BaseModel, Field

logger = getLogger(__name__)
handler = StreamHandler()
handler.setLevel(DEBUG)
logger.setLevel(DEBUG)
logger.addHandler(handler)

# 環境変数からAPIキーを読み込む
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
NUMBER_OF_EXAMPLES = os.getenv("NUMBER_OF_EXAMPLES", default=5)


from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)

# Gemini APIの初期化
llm = GoogleGenerativeAI(
    model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY, rate_limiter=rate_limiter
)


# 例文生成用のプロンプトテンプレート
example_prompt = PromptTemplate.from_template(
    """
    I am creating an English dictionary for language learners.  
Please explain the following English word in clear, simple English.  
Include all of the following sections:

1. Meaning (in easy English)
2. Nuance (how the word feels, when it is used, and how formal it is)
3. Example sentences (3 sentences, natural and useful)
4. Synonyms (with short notes about the differences)

Target audience: intermediate English learners (B1–B2).  
Avoid difficult vocabulary unless necessary.  
The word is: "{word}"
    
    example:

    # symposium

    ## Meaning (easy English)

    A symposium is a formal meeting where experts or specialists talk about a specific topic.
    People usually give presentations, share research, and discuss ideas with the audience.

    It is similar to a small academic conference, but usually more focused on one theme.

    ⸻

    ## Nuance
        •	Sounds academic, professional, and more formal than “meeting.”
        •	Often used in universities, research fields, science, medicine, or industry.
        •	Usually includes talks, panel discussions, and Q&A sessions.

    If you hear “symposium,” you can imagine a serious event where people come to learn and exchange knowledge.

    ⸻

    ## Example sentences
        •	Our department will host a symposium on renewable energy next month.
        •	Several scholars presented their latest findings at the international symposium.
        •	I attended a symposium about AI safety, and the discussions were very insightful.

    ⸻

    ## Synonyms (with nuance)
        •	conference — larger and broader; can cover many topics.
        •	seminar — usually smaller, more educational; often involves instruction.
        •	workshop — hands-on, participants actively practice skills.
        •	panel discussion — a group of speakers talking about a topic; often part of a symposium.
    """
)

# Chain
chain = example_prompt | llm | StrOutputParser()


def load_input_csv(input_file: str):
    # 入力CSVの読み込み(重複なしの想定)
    try:
        df = pd.read_csv(
            input_file, dtype={"word_id": int, "word": str}, na_filter=False
        )
    except Exception as e:
        print(f"CSVファイルの読み込みエラー: {e}")
        raise ValueError(f"CSVファイルの読み込みエラー: {e}")
    return df


def check_existing_words(output_file: str):
    """既存の出力ファイルをチェックし、完全に処理された単語を取得する"""

    fully_processed_words = set()
    word_explanation_dict = {}

    if not os.path.exists(output_file):
        return fully_processed_words, word_explanation_dict

    try:
        existing_df = pd.read_csv(
            output_file,
            dtype={"word_id": int, "word": str, "explanation": str},
            na_filter=False,
        )
        # NaNの処理を改善（pandasの標準的な方法でNaNをチェック）
        for index, row in existing_df.iterrows():
            word = row["word"]
            explanation = row["explanation"]
            if pd.notna(explanation) and explanation != "":
                fully_processed_words.add(word)
                word_explanation_dict[word] = explanation
        print(
            f"既存の出力ファイルから{len(fully_processed_words)}件のデータを読み込みました。"
        )
    except Exception as e:
        print(f"既存の出力ファイルの読み込みエラー: {e}")
    return fully_processed_words, word_explanation_dict


def generate_example(word: str) -> List[str]:
    """単語に対して例文を生成する"""
    # 例文の生成
    output = chain.invoke({"word": word})
    # 出力をパース
    examples: str = output  # .explanation
    return examples


def convert_to_long_format(results: List[dict], word_id_dict: dict) -> List[dict]:
    """結果をロング形式に変換する"""
    long_format_results = []
    for index, row in word_id_dict.iterrows():
        word = row["word"]
        word_id = row["word_id"]

        if word in results:
            long_format_results.append(
                {
                    "word_id": word_id,
                    "word": word,
                    "explanation": results[word],
                }
            )
    return long_format_results


def process_csv(input_file, output_file):
    """入力CSVファイルを処理し、各単語に例文を追加して新しいCSVに保存する"""
    df = load_input_csv(input_file)

    # 既存の出力ファイルをチェック（続きから処理するため）
    fully_processed_words, word_explanation_dict = check_existing_words(output_file)

    # 各単語を処理
    for index, row in df.iterrows():
        word = row["word"]

        # 既に処理済みの単語はスキップ
        if word in fully_processed_words:
            print(f"スキップ: {word} (既に処理済み)")
            continue
        print(f"処理中: {word} ({index+1}/{len(df)})")

        try:
            # 例文を生成
            examples = generate_example(word)
            # 結果を辞書に追加または更新
            word_explanation_dict[word] = examples
        except Exception as e:
            print(f"単語「{word}」の処理中にエラーが発生: {e}")
            # エラーが発生した場合も、空の例文で結果を追加して保存
            word_explanation_dict[word] = ""

        # 形式を変形し，ファイルに保存
        long_format_results = convert_to_long_format(word_explanation_dict, df)
        pd.DataFrame(long_format_results).to_csv(
            output_file, index=False, encoding="utf-8"
        )
        print(f"  → {word}の処理完了。中間結果を保存しました。")
        # APIリクエスト制限を考慮して少し待機
        time.sleep(5)

    print(f"処理が完了しました。結果は {output_file} に保存されています。")

In [ ]:
# 使用例
if __name__ == "__main__":
    # input_file = "../generate_examples/lv6.csv"  # 入力ファイル名
    input_file = "../word_data/originals/lv6.csv"
    output_file = "../explanation_data/english_lv6_explanation.csv"  # 出力ファイル名

    process_csv(input_file, output_file)